# Lending Club P2P Loan Default Prediction — Part 1: Baseline Reproduction

**UE24CS352A Machine Learning — Mini-Project, Team A-15 (Sec-A)**
Aman Priyansh (PES1UG24AM030) and Addanki Harsha Rajeswari (PES1UG24AM014)

Paper reproduced: *LendingClub Loan Default and Profitability Prediction*, Peiqian Li and Gao Han, Stanford CS229, 2018.

## 1. Data and the authors' code

In [ ]:
# CELL 1 - Get the data and the author's code
# 1) Download the Lending Club dataset from Kaggle (about 1.6 GB)
# 2) Download the paper's original code zip from the author's Google Drive
# We only READ the author's code to copy their steps. We do not run it.

import kagglehub, os, glob, zipfile, gdown

data_dir = kagglehub.dataset_download("wordsforthewise/lending-club")
print("Data folder:", data_dir)
for f in glob.glob(data_dir + "/**/*", recursive=True):
    print("  ", f, round(os.path.getsize(f)/1e6, 1), "MB")

gdown.download(id="1EilPoVCYx6fI84FUrfJvxGCUho5TImXx", output="author_code.zip", quiet=True)
with zipfile.ZipFile("author_code.zip") as z:
    z.extractall("author_code")
    print("Author files:", sorted(z.namelist()))

In [ ]:
# CELL 2 - Print the author's key code files so we can copy their exact steps
files = ["utils.py", "features_eng.py", "log_reg.py", "random_forest.py", "classification_eval.py",
         "utils_raw_return_regression.py", "ridge.py", "linear_regression.py", "mlp_regressor.py",
         "random_forest_regressor.py", "random_forest_shallow_regressor_10.py", "regression_eval.py", "regression_cost_eval.py"]
for name in files:
    print("=" * 30, name, "=" * 30)
    print(open("author_code/" + name).read())

## 2. Rebuild the missing cleaned dataset

In [ ]:
# CELL 3 - Rebuild the paper's "data_clean.csv"
# The author's code reads a file called data_clean.csv that is NOT in the zip.
# The paper (Section III-A) says how it was made:
#   - loans issued 2012 to 2015
#   - only loans with a FINAL status: "Fully Paid", "Charged Off", "Default"
#     (so "Current", "Late", "In Grace Period" are removed)
import pandas as pd, numpy as np

csv_file = data_dir + "/accepted_2007_to_2018q4.csv/accepted_2007_to_2018Q4.csv"

# All columns the author's classification + regression code uses, plus issue_d to filter by year
author_cols = ['loan_amnt','funded_amnt','total_pymnt','term','int_rate','installment','issue_d','last_pymnt_d',
 'emp_length','home_ownership','annual_inc','verification_status','loan_status','purpose','zip_code','addr_state','dti',
 'delinq_2yrs','earliest_cr_line','fico_range_low','fico_range_high','inq_last_6mths','mths_since_last_delinq',
 'mths_since_last_record','open_acc','pub_rec','revol_bal','revol_util','total_acc','out_prncp','out_prncp_inv',
 'last_fico_range_high','last_fico_range_low','collections_12_mths_ex_med','mths_since_last_major_derog','policy_code',
 'application_type','annual_inc_joint','dti_joint','acc_now_delinq','tot_coll_amt','tot_cur_bal','open_acc_6m',
 'open_act_il','open_il_12m','open_il_24m','mths_since_rcnt_il','total_bal_il','il_util','open_rv_12m','open_rv_24m',
 'all_util','total_rev_hi_lim','inq_fi','total_cu_tl','inq_last_12m','acc_open_past_24mths','avg_cur_bal',
 'bc_open_to_buy','bc_util','chargeoff_within_12_mths','delinq_amnt','mo_sin_old_il_acct','mo_sin_old_rev_tl_op',
 'mo_sin_rcnt_rev_tl_op','mo_sin_rcnt_tl','mort_acc','mths_since_recent_bc','mths_since_recent_bc_dlq',
 'mths_since_recent_inq','mths_since_recent_revol_delinq','num_accts_ever_120_pd','num_actv_bc_tl','num_actv_rev_tl',
 'num_bc_sats','num_bc_tl','num_il_tl','num_op_rev_tl','num_rev_accts','num_rev_tl_bal_gt_0','num_sats',
 'num_tl_120dpd_2m','num_tl_30dpd','num_tl_90g_dpd_24m','num_tl_op_past_12m','pct_tl_nvr_dlq','percent_bc_gt_75',
 'pub_rec_bankruptcies','tax_liens','tot_hi_cred_lim','total_bal_ex_mort','total_bc_limit','total_il_high_credit_limit']

raw = pd.read_csv(csv_file, usecols=author_cols, low_memory=False)
print("Full Kaggle file:", raw.shape)

raw["issue_year"] = pd.to_datetime(raw["issue_d"], format="%b-%Y").dt.year
clean = raw[raw["issue_year"].between(2012, 2015) &
            raw["loan_status"].isin(["Fully Paid", "Charged Off", "Default"])].copy()
clean = clean.drop(columns=["issue_year"])
del raw

print("After filter (paper had 745,529):", len(clean))
print(clean["loan_status"].value_counts())
print((clean["loan_status"] != "Fully Paid").mean().round(3), "= share negative (paper: 0.19)")
clean.to_csv("data_clean.csv", index=False)